# 03 · Inference
Load the packaged pipeline (preprocessing + model) and score new patients. Same artifact that the Docker image serves.

In [1]:
import os, pathlib
if pathlib.Path.cwd().name == 'notebooks':
    os.chdir('..')
import warnings; warnings.filterwarnings('ignore')
from IPython.display import Image, display

In [2]:
import json, joblib, pandas as pd
from heartops.data import FEATURES
model = joblib.load('artifacts/model.joblib')
patient = json.load(open('examples/patient.json'))
patient

{'age': 63,
 'sex': 1,
 'cp': 1,
 'trestbps': 145,
 'chol': 233,
 'fbs': 1,
 'restecg': 2,
 'thalach': 150,
 'exang': 0,
 'oldpeak': 2.3,
 'slope': 3,
 'ca': 0,
 'thal': 6}

In [3]:
p = model.predict_proba(pd.DataFrame([patient], columns=FEATURES))[0, 1]
{'prediction': int(p >= .5), 'disease_probability': round(p, 4), 'confidence': round(max(p, 1 - p), 4)}

{'prediction': 0,
 'disease_probability': np.float64(0.3809),
 'confidence': np.float64(0.6191)}

Missing `ca`/`thal` are imputed by the saved pipeline:

In [4]:
model.predict_proba(pd.DataFrame([{**patient, 'ca': None, 'thal': None}], columns=FEATURES, dtype=float))[0, 1]

np.float64(0.28047081754653186)

## Batch scoring with the CLI

In [5]:
import subprocess, sys
print(subprocess.run([sys.executable, 'scripts/predict.py', 'data/processed/heart.csv', '--limit', '3'], capture_output=True, text=True).stdout)

[
  {
    "prediction": 0,
    "disease_probability": 0.38090219387565344,
    "confidence": 0.6190978061243466
  },
  {
    "prediction": 1,
    "disease_probability": 0.9786699795572864,
    "confidence": 0.9786699795572864
  },
  {
    "prediction": 1,
    "disease_probability": 0.9686128409248618,
    "confidence": 0.9686128409248618
  }
]



## Calling the deployed API
With the container (`docker compose up`) or the Kubernetes ingress running:
```bash
curl -H 'Content-Type: application/json' --data @examples/patient.json http://localhost:8000/predict
python scripts/predict.py examples/patient.json --url http://localhost:8081
```